In [4]:
from pathlib import Path
import re
import pandas as pd

# Notebook is in: notebook/...
PROJECT_ROOT = Path("..").resolve()
ASSEMBLIES_ROOT = PROJECT_ROOT / "data" / "assemblies_spades"

print("PROJECT_ROOT     =", PROJECT_ROOT)
print("ASSEMBLIES_ROOT  =", ASSEMBLIES_ROOT)

def read_text(p: Path) -> str:
    try:
        return p.read_text(errors="ignore")
    except Exception:
        return ""

def fasta_contig_lengths(fa_path: Path):
    lens = []
    cur = 0
    for line in fa_path.read_text(errors="ignore").splitlines():
        if line.startswith(">"):
            if cur > 0:
                lens.append(cur)
                cur = 0
        else:
            cur += len(line.strip())
    if cur > 0:
        lens.append(cur)
    return lens

def n50_l50(lengths):
    if not lengths:
        return None, None
    lengths = sorted(lengths, reverse=True)
    total = sum(lengths)
    half = total / 2
    csum = 0
    for i, L in enumerate(lengths, start=1):
        csum += L
        if csum >= half:
            return L, i
    return lengths[-1], len(lengths)

def gfa_stats(gfa_path: Path):
    nodes = 0
    edges = 0
    for line in gfa_path.read_text(errors="ignore").splitlines():
        if line.startswith("S\t"):
            nodes += 1
        elif line.startswith("L\t"):
            edges += 1
    return nodes, edges

def find_gfa(run_root: Path):
    # SPAdes graph filenames vary by version
    for name in [
        "assembly_graph_with_scaffolds.gfa",
        "assembly_graph.gfa",
        "assembly_graph_with_scaffolds.gfa.gz",
        "assembly_graph.gfa.gz",
    ]:
        p = run_root / name
        if p.exists():
            return p
    return None

def parse_spades_log(run_root: Path):
    """
    Best-effort: SPAdes logs differ across versions.
    We'll search a few common places/files and patterns.
    """
    candidates = [
        run_root / "spades.log",
        run_root / "spades.log.txt",
        run_root / "params.txt",
    ]
    txt = ""
    for c in candidates:
        if c.exists():
            txt += "\n" + read_text(c)

    reads_pairs = None

    # Some possible patterns (not guaranteed):
    patterns = [
        r"Number of read-?pairs:\s*(\d+)",
        r"Total\s+pairs:\s*(\d+)",
        r"total\s+paired\s+reads:\s*(\d+)",
        r"paired\s+reads:\s*(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, txt, flags=re.I)
        if m:
            reads_pairs = int(m.group(1))
            break

    return {"reads_pairs_used": reads_pairs}

def summarize_spades_run(run_root: Path):
    contigs = run_root / "contigs.fasta"
    scaffolds = run_root / "scaffolds.fasta"

    if not contigs.exists():
        return None

    # contigs metrics
    lens = fasta_contig_lengths(contigs)
    n_contigs = len(lens)
    total_bp = sum(lens) if lens else 0
    max_len = max(lens) if lens else None
    n50, l50 = n50_l50(lens)

    # scaffolds metrics (optional)
    scaff_lens = fasta_contig_lengths(scaffolds) if scaffolds.exists() else []
    n_scaff = len(scaff_lens) if scaff_lens else None
    scaff_bp = sum(scaff_lens) if scaff_lens else None
    scaff_n50, scaff_l50 = n50_l50(scaff_lens) if scaff_lens else (None, None)

    # graph metrics (optional)
    gfa = find_gfa(run_root)
    nodes = edges = None
    if gfa and gfa.suffix == ".gz":
        # If you ever get .gfa.gz, easiest is to skip nodes/edges unless you want gzip parsing.
        # For now, we skip to avoid extra dependencies.
        nodes = edges = None
    elif gfa:
        nodes, edges = gfa_stats(gfa)

    # log metrics (optional)
    logm = parse_spades_log(run_root)

    return {
        "run_root": str(run_root),
        **logm,
        "contigs_fasta": str(contigs),
        "contigs_n": n_contigs,
        "contigs_bp": total_bp,
        "contigs_max": max_len,
        "contigs_N50": n50,
        "contigs_L50": l50,
        "scaffolds_fasta": str(scaffolds) if scaffolds.exists() else None,
        "scaffolds_n": n_scaff,
        "scaffolds_bp": scaff_bp,
        "scaffolds_N50": scaff_n50,
        "scaffolds_L50": scaff_l50,
        "graph_gfa": str(gfa) if gfa else None,
        "gfa_nodes": nodes,
        "gfa_edges": edges,
    }

def parse_folder_name(folder: str):
    """
    folder examples:
      10K_ext5_mixed_unfiltered
      10K_ext5_mixed_filtered
      10K_ext5_mixed_gb_filtered
      10K_ext5_mixed_cnn_filtered

    We'll infer:
      dataset = everything before the suffix
      filter  = unfiltered | filtered | gb | cnn
    """
    if folder.endswith("_unfiltered"):
        return folder[:-len("_unfiltered")], "unfiltered"
    if folder.endswith("_gb_filtered"):
        return folder[:-len("_gb_filtered")], "gb"
    if folder.endswith("_cnn_filtered"):
        return folder[:-len("_cnn_filtered")], "cnn"
    if folder.endswith("_filtered"):
        # ambiguous: could be GB-only, CNN-only, or GB->CNN sequential
        # label as "filtered" and you can later map it to mode via your run notes
        return folder[:-len("_filtered")], "filtered"
    return None, None

rows = []
if not ASSEMBLIES_ROOT.exists():
    raise FileNotFoundError(f"Missing folder: {ASSEMBLIES_ROOT}")

for run_root in sorted([p for p in ASSEMBLIES_ROOT.iterdir() if p.is_dir()]):
    dataset, flt = parse_folder_name(run_root.name)
    if dataset is None:
        continue  # ignore other folders if any

    rec = summarize_spades_run(run_root)
    if rec is None:
        rows.append({
            "dataset": dataset,
            "filter": flt,
            "status": "MISSING",
            "run_root": str(run_root),
        })
    else:
        rows.append({
            "dataset": dataset,
            "filter": flt,
            "status": "OK",
            **rec,
        })

df = pd.DataFrame(rows).sort_values(["dataset","filter"]).reset_index(drop=True)

missing = df[df["status"] != "OK"][["dataset","filter","run_root"]]
if len(missing):
    print("Missing runs (folder exists but contigs.fasta not found):")
    display(missing)

df

PROJECT_ROOT     = /Users/yvonnelin/Desktop/mitochime
ASSEMBLIES_ROOT  = /Users/yvonnelin/Desktop/mitochime/data/assemblies_spades


,dataset,filter,status,run_root,reads_pairs_used,contigs_fasta,contigs_n,contigs_bp,contigs_max,contigs_N50,contigs_L50,scaffolds_fasta,scaffolds_n,scaffolds_bp,scaffolds_N50,scaffolds_L50,graph_gfa,gfa_nodes,gfa_edges
0,10K_gibbosa10,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16657,16657,16657,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16657.0,16657.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
1,10K_gibbosa10,gb,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,6,4742,1617,1063,2,/Users/yvonnelin/Desktop/mitochime/data/assemb...,5.0,4752.0,2690.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,10,0
2,10K_gibbosa10,unfiltered,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16657,16657,16657,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16657.0,16657.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
3,10K_gibbosa15,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16658,16658,16658,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16658.0,16658.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
4,10K_gibbosa15,gb,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,5,5402,1628,1326,2,/Users/yvonnelin/Desktop/mitochime/data/assemb...,3.0,5466.0,2964.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,11,0
5,10K_gibbosa15,unfiltered,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16658,16658,16658,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16658.0,16658.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
6,10K_gibbosa5,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16654,16654,16654,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16654.0,16654.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
7,10K_gibbosa5,gb,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,7,4376,1241,1032,2,/Users/yvonnelin/Desktop/mitochime/data/assemb...,4.0,4601.0,2719.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,11,0
8,10K_gibbosa5,unfiltered,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16654,16654,16654,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16654.0,16654.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0
9,10K_gibbosa50,cnn,OK,/Users/yvonnelin/Desktop/mitochime/data/assemb...,None,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,16648,16648,16648,1,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1.0,16648.0,16648.0,1.0,/Users/yvonnelin/Desktop/mitochime/data/assemb...,1,0


In [6]:
import numpy as np
import pandas as pd

unf = df[(df["filter"] == "unfiltered") & (df["status"] == "OK")].copy()
flt = df[(df["filter"] != "unfiltered") & (df["status"] == "OK")].copy()

merged = flt.merge(
    unf[[
        "dataset",
        "reads_pairs_used",
        "contigs_n",
        "contigs_bp",
        "contigs_N50",
        "contigs_max",
        "gfa_nodes",
        "gfa_edges",
    ]],
    on="dataset",
    how="left",
    suffixes=("", "_unf"),
)

# --- force numeric (handles None/strings safely) ---
num_cols = [
    "reads_pairs_used", "reads_pairs_used_unf",
    "contigs_n", "contigs_n_unf",
    "contigs_bp", "contigs_bp_unf",
    "contigs_N50", "contigs_N50_unf",
    "contigs_max", "contigs_max_unf",
    "gfa_nodes", "gfa_nodes_unf",
    "gfa_edges", "gfa_edges_unf",
]
for c in num_cols:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

# reads kept % (only where both exist and denom != 0)
denom = merged["reads_pairs_used_unf"]
numer = merged["reads_pairs_used"]
reads_kept = (numer / denom) * 100
merged["reads_kept_pct"] = reads_kept.where(denom.notna() & (denom != 0) & numer.notna())
merged["reads_kept_pct"] = merged["reads_kept_pct"].round(2)

# deltas vs unfiltered
merged["delta_bp"]  = merged["contigs_bp"]  - merged["contigs_bp_unf"]
merged["delta_N50"] = merged["contigs_N50"] - merged["contigs_N50_unf"]
merged["delta_max"] = merged["contigs_max"] - merged["contigs_max_unf"]

results = (
    merged[[
        "dataset","filter",
        "reads_pairs_used_unf","reads_pairs_used","reads_kept_pct",
        "contigs_n_unf","contigs_n",
        "contigs_bp_unf","contigs_bp","delta_bp",
        "contigs_N50_unf","contigs_N50","delta_N50",
        "contigs_max_unf","contigs_max","delta_max",
        "gfa_nodes_unf","gfa_nodes",
        "gfa_edges_unf","gfa_edges",
        "run_root",
    ]]
    .rename(columns={
        "reads_pairs_used_unf": "reads_pairs_unfiltered",
        "reads_pairs_used": "reads_pairs_filtered",
        "contigs_n_unf": "contigs_unfiltered",
        "contigs_n": "contigs_filtered",
        "contigs_bp_unf": "bp_unfiltered",
        "contigs_bp": "bp_filtered",
        "contigs_N50_unf": "N50_unfiltered",
        "contigs_N50": "N50_filtered",
        "contigs_max_unf": "max_unfiltered",
        "contigs_max": "max_filtered",
        "gfa_nodes_unf": "nodes_unfiltered",
        "gfa_nodes": "nodes_filtered",
        "gfa_edges_unf": "edges_unfiltered",
        "gfa_edges": "edges_filtered",
    })
    .sort_values(["dataset","filter"])
    .reset_index(drop=True)
)

results

,dataset,filter,reads_pairs_unfiltered,reads_pairs_filtered,reads_kept_pct,contigs_unfiltered,contigs_filtered,bp_unfiltered,bp_filtered,delta_bp,...,N50_filtered,delta_N50,max_unfiltered,max_filtered,delta_max,nodes_unfiltered,nodes_filtered,edges_unfiltered,edges_filtered,run_root
0,10K_gibbosa10,cnn,NaN,NaN,NaN,1,1,16657,16657,0,...,16657,0,16657,16657,0,1,1,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
1,10K_gibbosa10,gb,NaN,NaN,NaN,1,6,16657,4742,-11915,...,1063,-15594,16657,1617,-15040,1,10,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
2,10K_gibbosa15,cnn,NaN,NaN,NaN,1,1,16658,16658,0,...,16658,0,16658,16658,0,1,1,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
3,10K_gibbosa15,gb,NaN,NaN,NaN,1,5,16658,5402,-11256,...,1326,-15332,16658,1628,-15030,1,11,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
4,10K_gibbosa5,cnn,NaN,NaN,NaN,1,1,16654,16654,0,...,16654,0,16654,16654,0,1,1,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
5,10K_gibbosa5,gb,NaN,NaN,NaN,1,7,16654,4376,-12278,...,1032,-15622,16654,1241,-15413,1,11,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
6,10K_gibbosa50,cnn,NaN,NaN,NaN,1,1,16648,16648,0,...,16648,0,16648,16648,0,1,1,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
7,10K_gibbosa50,gb,NaN,NaN,NaN,1,14,16648,7657,-8991,...,644,-16004,16648,1365,-15283,1,17,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
8,10K_lemuru10,cnn,NaN,NaN,NaN,1,1,16635,16522,-113,...,16522,-113,16635,16522,-113,1,1,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
9,10K_lemuru10,gb,NaN,NaN,NaN,1,1,16635,16614,-21,...,16614,-21,16635,16614,-21,1,1,0,0,/Users/yvonnelin/Desktop/mitochime/data/assemb...
